# Project 4: Automated CSV Explorer
Write a script that accepts any CSV path as a
command-line argument, auto-detects numeric
columns, generates summary statistics using
Pandas, plots histograms for all numeric
columns, and saves a self-contained HTML
report embedding the plots as base64 images.
Skills: sys.argv, Pandas, Matplotlib, io.BytesIO,
base64, HTML generation

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import base64
from io import BytesIO

def generate_automated_report(csv_path):
    """
    Core function to analyze CSV data and generate a base64-embedded HTML report.
    """
    #  LOAD DATA 
    try:
        df = pd.read_csv(csv_path)
        print(f"Processing: {csv_path}")
    except Exception as e:
        print(f"Error loading file: {e}")
        return

    # AUTO-DETECT NUMERIC COLUMNS 
    numeric_df = df.select_dtypes(include=['number'])
    if numeric_df.empty:
        print("No numeric columns found to analyze.")
        return

    #  SUMMARY STATISTICS 
    # Using .round(2) for professional formatting
    stats_html = numeric_df.describe().round(2).to_html(classes="table table-striped table-hover")

    # 4. PLOT HISTOGRAMS (Matplotlib)
    numeric_df.hist(figsize=(12, 10), bins=15, color='#3498db', edgecolor='white')
    plt.suptitle(f"Distribution Analysis: {csv_path}", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # EMBED IMAGE AS BASE64 (io.BytesIO, base64)
    buffer = BytesIO()
    plt.savefig(buffer, format='png')
    buffer.seek(0)
    img_base64 = base64.b64encode(buffer.read()).decode('utf-8')
    buffer.close()
    plt.close()

    #  HTML GENERATION (HTML generation)
    html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Data Explorer Report</title>
        <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
        <style>
            body {{ background-color: #f8f9fa; padding: 40px; }}
            .container {{ background: white; padding: 30px; border-radius: 15px; box-shadow: 0 0 20px rgba(0,0,0,0.1); }}
            h1 {{ color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }}
            img {{ max-width: 100%; height: auto; margin-top: 20px; border: 1px solid #dee2e6; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>📊 Automated Data Audit</h1>
            <p class="text-muted">Target File: <strong>{csv_path}</strong></p>
            
            <h3 class="mt-4">Summary Statistics</h3>
            <div class="table-responsive">
                {stats_html}
            </div>

            <h3 class="mt-5">Numeric Distributions</h3>
            <div class="text-center">
                <img src="data:image/png;base64,{img_base64}" alt="Data Histograms">
            </div>
            
            <footer class="mt-5 text-center text-secondary">
                Generated via Automated CSV Explorer | {pd.Timestamp.now().strftime('%Y-%m-%d')}
            </footer>
        </div>
    </body>
    </html>
    """

    # SAVE REPORT
    output_filename = "Explorer_Report.html"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print(f"Success! Self-contained report saved as: {output_filename}")

# COMMAND-LINE LOGIC (sys.argv)
if __name__ == "__main__":
   
    args = [a for a in sys.argv if not a.startswith('-')]
    
    if len(args) > 1:
        generate_automated_report(args[1])
    else:
        print("Usage: python explorer.py <sensor_data_sample.csv>")

Processing: C:\Users\SMARTECH\AppData\Roaming\jupyter\runtime\kernel-a0a74bf0-9c71-48f6-9b1c-f35362bfa4c6.json
Success! Self-contained report saved as: Explorer_Report.html
